# Netflix Data Analysis Using Python
Auspify Technologies internship — all six tasks. Run cells from top to bottom.

## 1. Import libraries and load dataset

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\n\ndf = pd.read_csv('Dataset.csv')\nprint('Shape:', df.shape)\ndf.head()

## 2. Inspect and clean data

In [ ]:
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]\nfor col in ['show_id','type','title','director','country','rating','duration','listed_in']:\n    df[col] = df[col].astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})\ndf['type'] = df['type'].str.title()\ndf['country'] = df['country'].str.replace(r'\\s+', ' ', regex=True)\ndf['rating'] = df['rating'].str.upper().str.strip()\ndf['listed_in'] = df['listed_in'].str.replace(r'\\s*,\\s*', ', ', regex=True)\ndf['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')\ndf['release_year'] = pd.to_numeric(df['release_year'], errors='coerce').astype('Int64')\nprint('Missing values before filling:\\n', df.isna().sum())\nprint('Exact duplicate rows:', df.duplicated().sum())\ndf = df.drop_duplicates()\ndf = df.drop_duplicates(subset=['show_id'], keep='first')\nfor col in ['director','country','rating','duration','listed_in']:\n    df[col] = df[col].fillna('Not Specified')\ndf.to_csv('netflix_cleaned.csv', index=False, date_format='%Y-%m-%d')\ndf.head()

## 3. Task 2 — Content type analysis

In [ ]:
type_counts = df['type'].value_counts()\nprint(type_counts)\nprint((type_counts / len(df) * 100).round(2))\ntype_counts.plot(kind='bar', title='Netflix Titles by Content Type')\nplt.xlabel('Content Type'); plt.ylabel('Number of Titles'); plt.tight_layout(); plt.show()

## 4. Task 3 — Country-wise analysis

In [ ]:
countries = df.assign(country_item=df['country'].str.split(', ')).explode('country_item')\ncountries = countries[countries['country_item'] != 'Not Specified']\ntop_countries = countries['country_item'].value_counts().head(10)\nprint(top_countries)\ntop_countries.sort_values().plot(kind='barh', title='Top 10 Countries by Netflix Titles')\nplt.xlabel('Title-country entries'); plt.tight_layout(); plt.show()

## 5. Task 4 — Release-year trends

In [ ]:
release_counts = df.dropna(subset=['release_year']).groupby('release_year').size().sort_index()\nrelease_counts[release_counts.index >= 2000].plot(kind='line', marker='o', title='Netflix Titles by Release Year (2000 onward)')\nplt.xlabel('Release Year'); plt.ylabel('Titles'); plt.tight_layout(); plt.show()\nadded = df['date_added'].dropna().dt.year.value_counts().sort_index()\nadded.plot(kind='line', marker='o', title='Netflix Titles Added by Year')\nplt.xlabel('Year Added'); plt.ylabel('Titles'); plt.tight_layout(); plt.show()

## 6. Task 5 — Ratings and genres

In [ ]:
print(df['rating'].value_counts().head(12))\ndf['rating'].value_counts().head(12).sort_values().plot(kind='barh', title='Most Common Ratings')\nplt.tight_layout(); plt.show()\ngenres = df.assign(genre_item=df['listed_in'].str.split(', ')).explode('genre_item')\ngenres = genres[genres['genre_item'] != 'Not Specified']\ntop_genres = genres['genre_item'].value_counts().head(15)\nprint(top_genres)\ntop_genres.sort_values().plot(kind='barh', title='Top 15 Genres')\nplt.tight_layout(); plt.show()\npd.crosstab(df['rating'], df['type']).loc[df['rating'].value_counts().head(10).index].plot(kind='bar', title='Rating by Content Type')\nplt.tight_layout(); plt.show()

## 7. Task 6 — Business insights

In [ ]:
total = len(df)\ncounts = df['type'].value_counts()\nprint(f'Total titles: {total:,}')\nfor kind, count in counts.items():\n    print(f'{kind}: {count:,} ({count/total:.1%})')\nprint('Top country entries:', top_countries.head(5).to_dict())\nprint('Top genre entries:', top_genres.head(5).to_dict())\nprint('Top ratings:', df['rating'].value_counts().head(5).to_dict())\nprint('Interpretation: country and genre columns are multi-label, so a title may count in multiple categories.')